# Módulo 02 · Indexación avanzada y `MultiIndex`

**Curso de pandas intermedio → avanzado** · Dataset: *Olist* (muestras en `data/`)

## Objetivos
1. Dominar `.loc`, `.iloc`, `.at` y máscaras booleanas sin ambigüedades.
2. Usar el **índice como llave**: `set_index`, unicidad, ordenamiento y búsquedas rápidas.
3. Entender la **alineación automática por índice** (la fuente de muchos `NaN` misteriosos).
4. Crear y navegar un **`MultiIndex`**: selección parcial, `pd.IndexSlice`, `.xs`.
5. Manipular niveles: `swaplevel`, `droplevel`, `rename_axis`, agregar por nivel.
6. Trabajar con **`MultiIndex` en columnas** y aplanarlas.
7. Usar un **`DatetimeIndex`** con selección por strings parciales.

| Sección | Tema |
|---|---|
| 0 | Setup y carga |
| 1 | Repaso de selección: `loc`, `iloc`, `at`, máscaras |
| 2 | El índice como llave |
| 3 | Alineación por índice |
| 4 | Crear un `MultiIndex` |
| 5 | Seleccionar en un `MultiIndex` |
| 6 | Manipular niveles |
| 7 | `MultiIndex` en columnas |
| 8 | `DatetimeIndex` |
| ✍️ | Ejercicios · ✅ Soluciones |

## 0 · Setup y carga

Cargamos sólo lo que usaremos, declarando los dtypes ahí mismo (repaso del Módulo 01).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("data")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)


def pendiente(*valores) -> bool:
  # True si algún valor sigue siendo `...` (ejercicio sin resolver)
  return any(v is ... for v in valores)


print("pandas", pd.__version__)

pandas 3.1.1


In [2]:
orders = pd.read_csv(
  DATA / "olist_orders_dataset.csv",
  dtype={"order_status": "category"},
  parse_dates=["order_purchase_timestamp", "order_approved_at",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
  usecols=lambda c: c != "order_delivered_carrier_date",
)

items = pd.read_csv(
  DATA / "olist_order_items_dataset.csv",
  dtype={"order_item_id": "int8"},
  parse_dates=["shipping_limit_date"],
)

products = pd.read_csv(
  DATA / "olist_products_dataset.csv",
  usecols=["product_id", "product_category_name", "product_weight_g"],
  dtype={"product_category_name": "category"},
)

sellers = pd.read_csv(
  DATA / "olist_sellers_dataset.csv",
  dtype={"seller_zip_code_prefix": "str", "seller_state": "category"},
)

{n: df.shape for n, df in [("orders", orders), ("items", items),
                           ("products", products), ("sellers", sellers)]}

{'orders': (300, 7),
 'items': (336, 7),
 'products': (296, 3),
 'sellers': (212, 4)}

## 1 · Repaso de selección: `loc`, `iloc`, `at`, máscaras

| Accesor | Selecciona por | Rebanada `a:b` |
|---|---|---|
| `.loc[filas, cols]` | **etiquetas** (o máscara booleana) | incluye `b` |
| `.iloc[filas, cols]` | **posiciones** enteras | excluye `b` |
| `.at[fila, col]` / `.iat[i, j]` | un solo valor (más rápido) | — |
| `df[col]` / `df[[cols]]` | columnas | — |
| `df[mascara]` | filas por máscara | — |

La diferencia en rebanadas es la fuente de muchos *off-by-one*:

In [3]:
demo = items.head(6)
print("loc[0:3]  ->", demo.loc[0:3].index.tolist())    # etiquetas 0..3 incluidas
print("iloc[0:3] ->", demo.iloc[0:3].index.tolist())   # posiciones 0..2

loc[0:3]  -> [0, 1, 2, 3]
iloc[0:3] -> [0, 1, 2]


In [4]:
# Filas y columnas a la vez: etiquetas en .loc
items.loc[items["price"] > 500, ["order_id", "price", "freight_value"]]

,order_id,price,freight_value
26,12aab7cb25bc6ae86c66cb3e7dc35a76,522.5,24.15
125,558731462aceb9588378c7c19194d2db,1054.0,35.96
186,8b50d4c9560cac1625c6fd9f536fc780,1700.0,30.87


### Construir máscaras legibles

Combina condiciones con `&`, `|`, `~` (¡siempre con paréntesis!). Métodos útiles: `isin`, `between`, `str.startswith`, `isna`.

In [5]:
caro = items["price"].between(200, 500)               # incluye ambos extremos por defecto
flete_alto = items["freight_value"] > 30
vendedores_top = items["seller_id"].isin(sellers.loc[sellers["seller_state"] == "PR", "seller_id"])

mascara = caro & (flete_alto | vendedores_top)
print("Filas que cumplen:", mascara.sum())
items.loc[mascara, ["price", "freight_value", "seller_id"]].head()

Filas que cumplen: 13


,price,freight_value,seller_id
36,269.99,32.92,d50d79cb34e38265a8649c383dcffd48
42,397.79,39.47,7b0df942f46435babab05d49b744b2c4
55,249.90,37.10,5bc24d989e71e93c33e50a7782431b0e
108,229.99,49.28,13d058e4eeac2ce8217660b2f8a05812
134,200.00,31.20,ad420dd0c4f92f8af951ac24b86d0cf5


`.at` es para leer/escribir **un** valor. Es la forma correcta (y rápida) cuando lo necesitas en un bucle:

In [6]:
print(items.at[0, "price"], items.iat[0, 5])

99.0 99.0


## 2 · El índice como llave

El `RangeIndex` (0, 1, 2…) no dice nada de tus datos. Poner una columna con significado como índice te da:

- Búsquedas por etiqueta con `.loc[llave]` (con índice único y ordenado son muy rápidas).
- Alineación automática al operar entre objetos (sección 3).
- Joins por índice (`join`, `map`).

In [7]:
orders_idx = orders.set_index("order_id")
assert orders_idx.index.is_unique, "order_id debería identificar cada pedido"
print("¿Único?   ", orders_idx.index.is_unique)
print("¿Ordenado?", orders_idx.index.is_monotonic_increasing)
orders_idx.head(3)

¿Único?    True
¿Ordenado? False


,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_customer_date,order_estimated_delivery_date
order_id,,,,,,
b9a6c5f5df52c7226ac85aee7524c27f,f160aaf480efdfa7268f0fa535f73e76,delivered,2018-06-12 20:07:44,2018-06-12 20:44:26,2018-06-19 12:44:08,2018-07-17
261e71d2349c713eafa9f3df5972b95d,d6708bbbd2d419475869a84e41f620a1,delivered,2018-01-20 12:15:57,2018-01-20 12:37:13,2018-01-30 11:32:35,2018-02-15
67b50899f52995848c427e361e10dde3,1b353c00c71689afba44554e43cc5a76,delivered,2018-06-16 21:24:10,2018-06-16 21:36:59,2018-06-27 13:17:27,2018-07-16


In [8]:
un_pedido = orders_idx.index[10]
orders_idx.loc[un_pedido]                 # un Series (una fila)

customer_id                      1c141ff3d234a4ea775feaf7d0024a59
order_status                                            delivered
order_purchase_timestamp                      2018-07-03 12:04:03
order_approved_at                             2018-07-05 16:32:43
order_delivered_customer_date                 2018-07-11 15:24:32
order_estimated_delivery_date                 2018-07-27 00:00:00
Name: 4fab84dd561f10bb9c98293f34d749fa, dtype: object

In [9]:
orders_idx.loc[[un_pedido]]               # lista de etiquetas -> DataFrame

,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_customer_date,order_estimated_delivery_date
order_id,,,,,,
4fab84dd561f10bb9c98293f34d749fa,1c141ff3d234a4ea775feaf7d0024a59,delivered,2018-07-03 12:04:03,2018-07-05 16:32:43,2018-07-11 15:24:32,2018-07-27


> ⚠️ **pandas 3.1:** el parámetro `set_index(..., verify_integrity=True)` está **deprecado**. El reemplazo recomendado es comprobar explícitamente `df.index.is_unique`, como en la celda anterior.

### Índices **no únicos**

`order_id` en `items` se repite (un pedido con varios ítems). `.loc` entonces devuelve **varias filas**, y el tipo del resultado depende de cuántas coincidan. Esto rompe código que asume siempre una fila:

In [10]:
items_by_order = items.set_index("order_id")
print("¿Único?", items_by_order.index.is_unique)

pedido_1_item = items_by_order.index[0]
pedido_6_items = items_by_order.index.value_counts().idxmax()

print(type(items_by_order.loc[pedido_1_item, "price"]).__name__)   # escalar
print(type(items_by_order.loc[pedido_6_items, "price"]).__name__)  # Series

¿Único? False
float64
Series


> 💡 Si necesitas siempre un DataFrame, usa una **lista**: `df.loc[[llave]]`.

### Ordenar el índice

Con un índice ordenado pandas usa búsqueda binaria en lugar de recorrer todo. En tablas grandes la diferencia es notable, y para algunas operaciones de `MultiIndex` es **obligatorio**.

In [11]:
items_by_order = items_by_order.sort_index()
items_by_order.index.is_monotonic_increasing

True

`reset_index` devuelve el índice a columna. Con `drop=True` lo descarta.

In [12]:
items_by_order.reset_index().head(2)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,008a40c2b8b21893e04945ce58854b55,1,016711f78a6a87696645db0a6a834826,a5cba26a62b8b4d0145b68b841e62e7f,2018-05-04 20:54:23,99.00,13.0
1,00fc308dd7f0937682698becaa9dcc45,1,940d38f0ef286bd93612fcdc91dd38ed,53e4c6e0f4312d4d2107a8c9cddf45cd,2018-08-23 20:10:12,85.71,18.4


## 3 · Alineación por índice

**Toda operación entre dos Series/DataFrames se alinea por etiquetas, no por posición.** Las etiquetas que faltan en alguno producen `NaN`.

Ejemplo: peso de cada producto vendido. `products` tiene el peso por `product_id`; `items` tiene ventas.

In [13]:
peso = products.set_index("product_id")["product_weight_g"]
peso.head(3)

product_id
c6336fa91fbd87c359e44f5dca5a90ed     150.0
00a985c524adbb97a4211e4ce17aabec     250.0
3932e37f1f658768ac53fbe680f145bc    1100.0
Name: product_weight_g, dtype: float64

In [14]:
# ❌ Sin índice común: se alinea por 0,1,2... ¡asigna el peso de OTRO producto!
mal = items["price"] / products["product_weight_g"]
print("Longitud:", len(mal), "| NaN:", mal.isna().sum())

Longitud: 336 | NaN: 40


El cálculo anterior "funcionó" sin error y es **incorrecto**: la fila 0 de `items` se dividió entre la fila 0 de `products`, que es otro producto. Además, como las longitudes difieren, aparecen `NaN` en el sobrante.

✅ La forma correcta es **traer** el valor usando la llave: `Series.map` con una Series indexada actúa como un *lookup*.

In [15]:
items_con_peso = items.assign(peso_g=items["product_id"].map(peso))
items_con_peso[["product_id", "price", "peso_g"]].head()

,product_id,price,peso_g
0,016711f78a6a87696645db0a6a834826,99.00,1150.0
1,940d38f0ef286bd93612fcdc91dd38ed,85.71,4300.0
2,0bcc3eeca39e1064258aa1e932269894,49.90,1800.0
3,927225847efa8853ebf52c9b5b9b8cc6,59.99,850.0
4,40df4be4429553e18755913ac3cfeac5,59.90,1450.0


### Alineación como herramienta

Cuando **ambos** lados están indexados por la misma llave, la alineación hace exactamente lo que quieres, aunque el orden y el tamaño difieran.

In [16]:
ventas_por_vendedor = items.groupby("seller_id")["price"].sum()
flete_por_vendedor = items.groupby("seller_id")["freight_value"].sum()

# Desordenamos uno a propósito: el resultado sigue siendo correcto
ratio = flete_por_vendedor.sample(frac=1, random_state=0) / ventas_por_vendedor
ratio.sort_values(ascending=False).head()

seller_id
609e1a9a6c2539919b8205cf7c4e6ff0    2.792405
db4350fd57ae30082dec7acbaacc17f9    2.135710
2709af9587499e95e803a6498a5a56e9    0.931500
4e17c65a516f69d023a2ae78b84f28d6    0.808966
8b321bb669392f5163d04c59e235e066    0.795780
dtype: float64

### `reindex`: conformar a un índice dado

`reindex` reordena según una lista de etiquetas e inserta `NaN` (o `fill_value`) en las que no existen. Útil para garantizar que aparezcan todas las categorías o todas las fechas.

In [17]:
estados_brasil = ["SP", "RJ", "MG", "PR", "SC", "RS", "BA", "AM"]
vendedores_por_estado = sellers["seller_state"].value_counts()

vendedores_por_estado.reindex(estados_brasil, fill_value=0)

seller_state
SP    140
RJ      6
MG     20
PR     26
SC      9
RS      3
BA      1
AM      0
Name: count, dtype: int64

## 4 · Crear un `MultiIndex`

Un `MultiIndex` es un índice con **varios niveles**: cada etiqueta es una tupla. Aparece naturalmente cuando:

- La llave natural es compuesta: un ítem se identifica por `(order_id, order_item_id)`.
- Agrupas por varias columnas.
- Haces `stack`/`unstack`/`pivot` (Módulo 04).

In [18]:
items_mi = items.set_index(["order_id", "order_item_id"]).sort_index()
assert items_mi.index.is_unique
items_mi.head(8)

,,product_id,seller_id,shipping_limit_date,price,freight_value
order_id,order_item_id,,,,,
008a40c2b8b21893e04945ce58854b55,1,016711f78a6a87696645db0a6a834826,a5cba26a62b8b4d0145b68b841e62e7f,2018-05-04 20:54:23,99.00,13.00
00fc308dd7f0937682698becaa9dcc45,1,940d38f0ef286bd93612fcdc91dd38ed,53e4c6e0f4312d4d2107a8c9cddf45cd,2018-08-23 20:10:12,85.71,18.40
0124e0c5bfae1979838a91e47df53f14,1,0bcc3eeca39e1064258aa1e932269894,1f50f920176fa81dab994f9023523100,2018-02-14 16:31:36,49.90,12.48
042649285cd08ef7c546445419b43407,1,927225847efa8853ebf52c9b5b9b8cc6,855668e0971d4dfd7bef1b6a4133b41b,2018-03-28 15:35:36,59.99,18.35
0452166f3b0d8e378179ecea1efdfe05,1,40df4be4429553e18755913ac3cfeac5,ff1e15b778c700abdd4d239b81ac466d,2018-08-14 11:35:23,59.90,11.91
0510f632b86f01064ebbde67deb63bb5,1,10070e29929f7c7d609ee3201f236b8e,8a130737016f838139d31878787a39c9,2018-06-29 00:31:25,95.20,18.47
0589d8b975f21fea2ad626ea99527911,1,ba01cebc5a3c2494d0a97d4713ec8938,a416b6a846a11724393025641d4edd5e,2018-03-06 16:31:15,201.34,25.96
06aa0aa8c22027ebb953d501c4238512,1,cd2c0529e45020c77a7c05b4ef2408fd,0885aaf116795758dfeb5f1032487bcd,2018-05-02 20:31:06,69.90,8.88


In [19]:
idx = items_mi.index
print("Niveles   :", idx.nlevels)
print("Nombres   :", idx.names)
print("Únicos por nivel:", [len(l) for l in idx.levels])
print("Primera etiqueta:", idx[0])

Niveles   : 2
Nombres   : ['order_id', 'order_item_id']
Únicos por nivel: [299, 6]
Primera etiqueta: ('008a40c2b8b21893e04945ce58854b55', np.int8(1))


Desde un `groupby` con varias llaves:

In [20]:
items_cat = items.assign(
  categoria=items["product_id"].map(products.set_index("product_id")["product_category_name"]).astype("str"),
  estado_vendedor=items["seller_id"].map(sellers.set_index("seller_id")["seller_state"]).astype("str"),
)

ventas = (
  items_cat.groupby(["estado_vendedor", "categoria"], observed=True)["price"]
  .sum()
  .sort_index()
)
ventas.head(10)

estado_vendedor  categoria                  
BA               cool_stuff                     229.99
DF               fashion_bolsas_e_acessorios    199.90
                 informatica_acessorios         319.00
ES               ferramentas_jardim             297.00
GO               automotivo                     112.33
                 bebes                          172.33
                 utilidades_domesticas          179.00
MG               automotivo                     249.90
                 bebes                          229.48
                 beleza_saude                   219.99
Name: price, dtype: float64

> ⚠️ **¿Por qué el `.astype("str")`?** Si los niveles fueran `category`, las rebanadas por rango (`"a":"c"`) seguirían el **orden de las categorías**, no el alfabético, y fallarían con etiquetas que no son categorías existentes. Para índices que vas a rebanar por texto, conviene usar `str`.

Y construido a mano (útil para plantillas o para `reindex`):

In [21]:
pd.MultiIndex.from_product(
  [["SP", "RJ"], ["2018-01", "2018-02", "2018-03"]],
  names=["estado", "mes"],
)

MultiIndex([('SP', '2018-01'),
            ('SP', '2018-02'),
            ('SP', '2018-03'),
            ('RJ', '2018-01'),
            ('RJ', '2018-02'),
            ('RJ', '2018-03')],
           names=['estado', 'mes'])

## 5 · Seleccionar en un `MultiIndex`

Usaremos la Series `ventas` (índice `estado_vendedor` × `categoria`).

| Quiero… | Sintaxis |
|---|---|
| Un elemento exacto | `s.loc[("SP", "telefonia")]` |
| Todo un valor del 1er nivel | `s.loc["SP"]` (selección parcial) |
| Varios valores del 1er nivel | `s.loc[["SP", "PR"]]` |
| Filtrar en un nivel interior | `s.loc[pd.IndexSlice[:, "telefonia"]]` o `s.xs("telefonia", level=1)` |
| Rangos en varios niveles | `s.loc[pd.IndexSlice["MG":"PR", "a":"c"]]` |
| Máscara por nivel | `s[s.index.get_level_values("categoria").str.startswith("moveis")]` |

In [22]:
ventas.loc[("SP", "telefonia")]

np.float64(441.21999999999997)

In [23]:
ventas.loc["PR"].sort_values(ascending=False).head()    # el nivel 0 desaparece

categoria
moveis_decoracao          714.79
beleza_saude              698.89
informatica_acessorios    556.79
esporte_lazer             398.90
perfumaria                313.90
Name: price, dtype: float64

In [24]:
ventas.loc[["RJ", "SC"]]                                 # el nivel 0 se conserva

estado_vendedor  categoria             
RJ               automotivo                  90.00
                 brinquedos                 299.99
                 eletrodomesticos          1054.00
                 moveis_decoracao            24.90
                 moveis_sala                 69.99
                 relogios_presentes        1700.00
SC               automotivo                  29.90
                 beleza_saude                19.90
                 brinquedos                  89.99
                 cama_mesa_banho             47.40
                 eletrodomesticos_2         522.50
                 esporte_lazer               42.90
                 informatica_acessorios      74.80
                 perfumaria                  40.90
                 relogios_presentes          35.90
Name: price, dtype: float64

### `pd.IndexSlice`

Para rebanar niveles interiores con `.loc` necesitas `slice(None)` (equivalente a `:`). `pd.IndexSlice` permite escribirlo con la sintaxis natural de rebanadas:

In [25]:
idx = pd.IndexSlice
ventas.loc[idx[:, "telefonia"]]

estado_vendedor
MG    274.99
SP    441.22
Name: price, dtype: float64

In [26]:
# Rangos en ambos niveles. Requiere índice ORDENADO (por eso hicimos sort_index)
ventas.loc[idx["MG":"PR", "a":"c"]]

estado_vendedor  categoria   
MG               automotivo      249.90
                 bebes           229.48
                 beleza_saude    219.99
                 brinquedos       64.99
PR               bebes           296.80
                 beleza_saude    698.89
Name: price, dtype: float64

> ⚠️ Si el `MultiIndex` no está ordenado, las rebanadas por rango lanzan `UnsortedIndexError` o son lentas. Cuando dudes: `df = df.sort_index()`.

### `.xs` (*cross-section*)

`xs` selecciona un valor en **cualquier nivel** y, por defecto, elimina ese nivel. `drop_level=False` lo conserva.

In [27]:
ventas.xs("cama_mesa_banho", level="categoria")

estado_vendedor
MG      69.90
SC      47.40
SP    2517.96
Name: price, dtype: float64

In [28]:
ventas.xs("cama_mesa_banho", level="categoria", drop_level=False).head(3)

estado_vendedor  categoria      
MG               cama_mesa_banho      69.90
SC               cama_mesa_banho      47.40
SP               cama_mesa_banho    2517.96
Name: price, dtype: float64

### Máscaras sobre un nivel con `get_level_values`

In [29]:
cat = ventas.index.get_level_values("categoria")
ventas[cat.str.startswith("moveis")]

estado_vendedor  categoria        
PR               moveis_decoracao     714.79
                 moveis_quarto        129.00
RJ               moveis_decoracao      24.90
                 moveis_sala           69.99
SP               moveis_decoracao     698.33
                 moveis_escritorio    849.91
Name: price, dtype: float64

En un DataFrame con `MultiIndex` todo lo anterior funciona igual, con columnas como segundo argumento de `.loc`:

In [30]:
pedido = items_mi.index.get_level_values("order_id").value_counts().idxmax()
items_mi.loc[pedido, ["price", "freight_value"]]

,price,freight_value
order_item_id,,
1,7.9,22.06
2,7.9,22.06
3,7.9,22.06
4,7.9,22.06
5,7.9,22.06
6,7.9,22.06


In [31]:
items_mi.loc[(pedido, 2), "price"]          # ítem 2 de ese pedido

np.float64(7.9)

## 6 · Manipular niveles

| Método | Hace |
|---|---|
| `swaplevel(i, j)` | intercambia dos niveles |
| `reorder_levels([...])` | reordena todos |
| `droplevel(n)` | elimina un nivel |
| `rename_axis([...])` / `index.set_names` | renombra niveles |
| `groupby(level=...)` | agrega por nivel(es) |
| `sort_index(level=...)` | ordena por nivel |

In [32]:
por_categoria = ventas.swaplevel().sort_index()
por_categoria.head()

categoria   estado_vendedor
alimentos   SP                 156.97
audio       SP                 137.20
automotivo  GO                 112.33
            MG                 249.90
            RJ                  90.00
Name: price, dtype: float64

In [33]:
ventas.rename_axis(["estado", "cat"]).head(3)

estado  cat                        
BA      cool_stuff                     229.99
DF      fashion_bolsas_e_acessorios    199.90
        informatica_acessorios         319.00
Name: price, dtype: float64

In [34]:
# Total por estado: agregar sobre un nivel
ventas.groupby(level="estado_vendedor", observed=True).sum().sort_values(ascending=False).head()

estado_vendedor
SP    21984.04
MG     4142.20
PR     3540.17
RJ     3238.88
SC      904.19
Name: price, dtype: float64

Un patrón muy común: **participación dentro del grupo**. Dividimos cada valor entre el total de su nivel superior con `transform` (lo veremos a fondo en el Módulo 03):

In [35]:
participacion = ventas / ventas.groupby(level="estado_vendedor", observed=True).transform("sum")
participacion.loc["SP"].sort_values(ascending=False).head().round(3)

categoria
cama_mesa_banho           0.115
esporte_lazer             0.110
beleza_saude              0.081
automotivo                0.069
informatica_acessorios    0.050
Name: price, dtype: float64

Y cuando quieres "abrir" un nivel a columnas, `unstack` (adelanto del Módulo 04):

In [36]:
ventas.loc[["SP", "PR", "MG"]].unstack("estado_vendedor", fill_value=0).head()

estado_vendedor,SP,PR,MG
categoria,,,
alimentos,156.97,0.00,0.00
audio,137.20,0.00,0.00
automotivo,1527.59,0.00,249.90
bebes,618.75,296.80,229.48
beleza_saude,1776.89,698.89,219.99


## 7 · `MultiIndex` en columnas

Aparece al agregar varias columnas con varias funciones:

In [37]:
resumen = items_cat.groupby("estado_vendedor", observed=True)[["price", "freight_value"]].agg(["sum", "mean", "count"])
resumen.head()

price                   freight_value                 
                     sum        mean count           sum       mean count
estado_vendedor                                                          
BA                229.99  229.990000     1         26.06  26.060000     1
DF                518.90  259.450000     2         63.70  31.850000     2
ES                297.00   99.000000     3         99.24  33.080000     3
GO                463.66  154.553333     3         74.73  24.910000     3
MG               4142.20  125.521212    33        732.72  22.203636    33

In [38]:
resumen.columns

MultiIndex([(        'price',   'sum'),
            (        'price',  'mean'),
            (        'price', 'count'),
            ('freight_value',   'sum'),
            ('freight_value',  'mean'),
            ('freight_value', 'count')],
           )

In [39]:
resumen.loc[:, ("price", "mean")].head(3)           # una columna exacta

estado_vendedor
BA    229.99
DF    259.45
ES     99.00
Name: (price, mean), dtype: float64

In [40]:
resumen.loc[:, idx[:, "mean"]].head(3)               # todas las "mean"

,price,freight_value
,mean,mean
estado_vendedor,,
BA,229.99,26.06
DF,259.45,31.85
ES,99.00,33.08


### Aplanar columnas

Para exportar o seguir trabajando, suele convenir aplanar a nombres simples:

In [41]:
plano = resumen.copy()
plano.columns = ["_".join(col) for col in plano.columns]
plano.head(3)

,price_sum,price_mean,price_count,freight_value_sum,freight_value_mean,freight_value_count
estado_vendedor,,,,,,
BA,229.99,229.99,1,26.06,26.06,1
DF,518.90,259.45,2,63.70,31.85,2
ES,297.00,99.00,3,99.24,33.08,3


> 💡 En el Módulo 03 veremos las **agregaciones con nombre** (`agg(precio_total=("price", "sum"))`), que evitan el `MultiIndex` de columnas desde el principio.

## 8 · `DatetimeIndex`

Con fechas en el índice (y **ordenadas**) puedes seleccionar con strings parciales: un año, un mes, un día o un rango.

In [42]:
orders_t = orders.set_index("order_purchase_timestamp").sort_index()
orders_t.index

DatetimeIndex(['2017-01-28 20:09:18', '2017-02-07 10:26:05', '2017-02-08 14:36:40', '2017-02-08 14:42:14', '2017-02-15 16:18:40',
               '2017-02-18 16:48:35', '2017-02-27 11:33:35', '2017-03-01 10:46:15', '2017-03-02 07:28:36', '2017-03-02 11:53:33',
               ...
               '2018-08-15 20:51:44', '2018-08-16 11:22:55', '2018-08-17 16:53:48', '2018-08-17 19:00:16', '2018-08-18 21:36:49',
               '2018-08-19 14:59:34', '2018-08-19 22:12:22', '2018-08-20 19:51:32', '2018-08-22 09:17:07', '2018-08-26 21:01:53'],
              dtype='datetime64[us]', name='order_purchase_timestamp', length=300, freq=None)

In [43]:
print("2018      :", len(orders_t.loc["2018"]))
print("2018-03   :", len(orders_t.loc["2018-03"]))
print("Q1 2018   :", len(orders_t.loc["2018-01":"2018-03"]))     # ambos extremos incluidos

2018      : 184
2018-03   : 23
Q1 2018   : 81


In [44]:
orders_t.loc["2018-03-01":"2018-03-05", ["order_status", "order_id"]]

,order_status,order_id
order_purchase_timestamp,,
2018-03-01 08:07:37,delivered,5f4942fc3a59f2e62ae534f981105954
2018-03-01 15:25:54,delivered,37c8cbbb816470ceb8a3026653cf3d3c
2018-03-03 21:40:09,delivered,b4c45d4cc2a2e24e10d4e20bae928819
2018-03-05 19:13:59,delivered,ec34ce0f06a9a35ebf0f0496d595bc84


Los atributos del índice (`.year`, `.month`, `.dayofweek`, `.hour`…) sirven para máscaras sin crear columnas:

In [45]:
nocturnos = orders_t[orders_t.index.hour < 6]
print("Pedidos de madrugada:", len(nocturnos))

Pedidos de madrugada: 11


> En el Módulo 06 llevaremos esto a `resample`, ventanas móviles y zonas horarias.

---
# ✍️ Ejercicios

Variables disponibles: `orders`, `items`, `products`, `sellers`, `items_cat`, `idx` (= `pd.IndexSlice`).

### Ejercicio 1 · Llave compuesta ⭐

1. Crea `im`: `items` indexado por `(order_id, order_item_id)`, ordenado, y verifica con un `assert` que la llave sea única.
2. `n_multi`: número de pedidos con **más de un** ítem. Usa el índice (pista: `get_level_values` o `groupby(level=...)`).
3. `segundo_item`: el `price` del ítem `2` del pedido `"f97b2bdfbaca9c2f04722d8cbc8cc8a5"`, usando **un solo** `.loc`.
4. `total_pedido`: Series con la suma de `price + freight_value` por `order_id`, calculada agregando sobre el nivel `order_id` de `im`.

In [46]:
# TU CÓDIGO AQUÍ
im = ...
n_multi = ...
segundo_item = ...
total_pedido = ...

In [47]:
# Comprobación
if pendiente(im, n_multi, segundo_item, total_pedido):
  print("⏳ Ejercicio 1 pendiente: resuélvelo en la celda de arriba")
else:
  assert isinstance(im.index, pd.MultiIndex) and im.index.names == ["order_id", "order_item_id"]
  assert im.index.is_monotonic_increasing, "Ordena el índice"
  assert n_multi == 27, f"n_multi = {n_multi}"
  assert np.isclose(segundo_item, 43.0), f"segundo_item = {segundo_item}"
  assert total_pedido.index.name == "order_id" and len(total_pedido) == items["order_id"].nunique()
  assert np.isclose(total_pedido.sum(), (items["price"] + items["freight_value"]).sum())
  print("✅ Ejercicio 1 correcto")

⏳ Ejercicio 1 pendiente: resuélvelo en la celda de arriba


### Ejercicio 2 · Lookups con `map` en lugar de merge ⭐

Sin usar `merge` ni `join`:

1. Crea `it` a partir de `items` con dos columnas nuevas: `estado_cliente` (estado del **cliente** que hizo el pedido) y `estado_vendedor`.
   Pistas: el cliente está en `orders` vía `customer_id`, y su estado está en `olist_customers_dataset.csv` (cárgalo tú con el dtype que creas conveniente). Necesitarás encadenar dos `map`.
2. `pct_mismo_estado`: porcentaje (0–100) de ítems donde el vendedor y el cliente están en el mismo estado.

In [48]:
# TU CÓDIGO AQUÍ
it = ...
pct_mismo_estado = ...

In [49]:
# Comprobación
if pendiente(it, pct_mismo_estado):
  print("⏳ Ejercicio 2 pendiente: resuélvelo en la celda de arriba")
else:
  assert {"estado_cliente", "estado_vendedor"} <= set(it.columns)
  assert len(it) == len(items), "No debes cambiar el número de filas"
  assert it["estado_cliente"].notna().all() and it["estado_vendedor"].notna().all()
  assert np.isclose(pct_mismo_estado, 33.9286, atol=0.01), f"pct_mismo_estado = {pct_mismo_estado}"
  print("✅ Ejercicio 2 correcto")

⏳ Ejercicio 2 pendiente: resuélvelo en la celda de arriba


### Ejercicio 3 · Navegando un `MultiIndex` ⭐⭐

Usa la Series `ventas` (índice `estado_vendedor` × `categoria`) de la sección 4.

1. `ventas_sp_pr`: ventas de SP y PR, sólo en categorías alfabéticamente entre `"b"` y `"e"` (rebanada `"b":"e"`). Usa `pd.IndexSlice`.
2. `cama_por_estado`: ventas de `"cama_mesa_banho"` por estado (índice = estado), con `.xs`.
3. `top_por_estado`: Series indexada por estado con el **nombre de la categoría** que más vende en cada estado.
   Pista: `groupby(level=...)` + `idxmax` devuelve tuplas; extrae el segundo elemento.

In [50]:
# TU CÓDIGO AQUÍ
ventas_sp_pr = ...
cama_por_estado = ...
top_por_estado = ...

In [51]:
# Comprobación
if pendiente(ventas_sp_pr, cama_por_estado, top_por_estado):
  print("⏳ Ejercicio 3 pendiente: resuélvelo en la celda de arriba")
else:
  assert set(ventas_sp_pr.index.get_level_values(0)) <= {"SP", "PR"}
  cats = ventas_sp_pr.index.get_level_values(1)
  assert ((cats >= "b") & (cats <= "e")).all()
  assert len(ventas_sp_pr) == 13, f"len = {len(ventas_sp_pr)}"
  assert cama_por_estado.index.name == "estado_vendedor"
  assert np.isclose(cama_por_estado.sum(), 2635.26)
  assert top_por_estado.loc["SP"] == "cama_mesa_banho", top_por_estado.loc["SP"]
  assert len(top_por_estado) == ventas.index.get_level_values(0).nunique()
  print("✅ Ejercicio 3 correcto")

⏳ Ejercicio 3 pendiente: resuélvelo en la celda de arriba


### Ejercicio 4 · `DatetimeIndex` ⭐⭐

Con `orders_t` (pedidos indexados por fecha de compra):

1. `n_bf`: pedidos hechos en la semana de Black Friday 2017, del **2017-11-20 al 2017-11-26** inclusive.
2. `por_trimestre`: Series con el número de pedidos de **cada trimestre de 2018** presente en los datos, con índice `"2018Q1"`, `"2018Q2"`, … como strings.
   Pista: `orders_t.index.to_period("Q")`.
3. `pct_fin_semana`: porcentaje (0–100) de pedidos hechos en sábado o domingo.

In [52]:
# TU CÓDIGO AQUÍ
n_bf = ...
por_trimestre = ...
pct_fin_semana = ...

In [53]:
# Comprobación
if pendiente(n_bf, por_trimestre, pct_fin_semana):
  print("⏳ Ejercicio 4 pendiente: resuélvelo en la celda de arriba")
else:
  assert n_bf == 7, f"n_bf = {n_bf}"
  assert list(por_trimestre.index) == ["2018Q1", "2018Q2", "2018Q3"], list(por_trimestre.index)
  assert por_trimestre.sum() == len(orders_t.loc["2018"])
  assert np.isclose(pct_fin_semana, 22.0, atol=0.01), f"pct_fin_semana = {pct_fin_semana}"
  print("✅ Ejercicio 4 correcto")

⏳ Ejercicio 4 pendiente: resuélvelo en la celda de arriba


### Ejercicio 5 · Reto: columnas jerárquicas ⭐⭐⭐

1. A partir de `items_cat`, crea `res` agrupando por `estado_vendedor` y agregando `price` y `freight_value` con `["sum", "mean"]`.
2. Aplana las columnas a `price_sum`, `price_mean`, `freight_value_sum`, `freight_value_mean`.
3. Agrega la columna `flete_relativo = freight_value_mean / price_mean`.
4. `estados_flete_caro`: lista **ordenada alfabéticamente** de estados (como strings) cuyo `flete_relativo` sea mayor a `0.25`.
5. **Bonus:** recupera el `MultiIndex` de columnas a partir de los nombres planos con `str.rsplit("_", n=1, expand=True)` y guárdalo en `res_mi`.

In [54]:
# TU CÓDIGO AQUÍ
res = ...
estados_flete_caro = ...
res_mi = ...

In [55]:
# Comprobación
if pendiente(res, estados_flete_caro, res_mi):
  print("⏳ Ejercicio 5 pendiente: resuélvelo en la celda de arriba")
else:
  assert {"price_sum", "price_mean", "freight_value_sum", "freight_value_mean", "flete_relativo"} <= set(res.columns)
  assert not isinstance(res.columns, pd.MultiIndex)
  assert estados_flete_caro == ["ES", "MT", "SC"], estados_flete_caro
  assert isinstance(res_mi.columns, pd.MultiIndex)
  assert ("freight_value", "mean") in res_mi.columns
  print("✅ Ejercicio 5 correcto")

⏳ Ejercicio 5 pendiente: resuélvelo en la celda de arriba


---
# ✅ Soluciones

### Solución 1

In [56]:
im = items.set_index(["order_id", "order_item_id"]).sort_index()
assert im.index.is_unique

items_por_pedido = im.groupby(level="order_id").size()
n_multi = int((items_por_pedido > 1).sum())

segundo_item = im.loc[("f97b2bdfbaca9c2f04722d8cbc8cc8a5", 2), "price"]

total_pedido = (im["price"] + im["freight_value"]).groupby(level="order_id").sum()
n_multi, segundo_item, total_pedido.head(3)

(27,
 np.float64(43.0),
 order_id
 008a40c2b8b21893e04945ce58854b55    112.00
 00fc308dd7f0937682698becaa9dcc45    104.11
 0124e0c5bfae1979838a91e47df53f14     62.38
 dtype: float64)

*Notas:* el `assert im.index.is_unique` es una aserción gratuita sobre tu modelo de datos: si algún día llegan duplicados, te enteras de inmediato. Como la llave `(order_id, order_item_id)` es única, `.loc[(pedido, 2), "price"]` devuelve un escalar. Alternativa para `n_multi`: `im.index.get_level_values(0).value_counts().gt(1).sum()`.

### Solución 2

In [57]:
customers = pd.read_csv(
  DATA / "olist_customers_dataset.csv",
  usecols=["customer_id", "customer_state"],
  dtype={"customer_state": "category"},
)

cliente_de_pedido = orders.set_index("order_id")["customer_id"]
estado_de_cliente = customers.set_index("customer_id")["customer_state"]
estado_de_vendedor = sellers.set_index("seller_id")["seller_state"]

it = items.assign(
  estado_cliente=items["order_id"].map(cliente_de_pedido).map(estado_de_cliente),
  estado_vendedor=items["seller_id"].map(estado_de_vendedor),
)

pct_mismo_estado = (
  it["estado_cliente"].astype("str") == it["estado_vendedor"].astype("str")
).mean() * 100
round(pct_mismo_estado, 2)

np.float64(33.93)

*Notas:* `map` con una Series indexada es un *lookup*: devuelve un resultado con el **mismo largo** que el original (a diferencia de un `merge`, que puede multiplicar filas si la llave se repite). Convertimos a `str` antes de comparar porque dos columnas `category` con **distintas** listas de categorías no se pueden comparar directamente.

### Solución 3

In [58]:
ventas_sp_pr = ventas.loc[idx[["PR", "SP"], "b":"e"]]

cama_por_estado = ventas.xs("cama_mesa_banho", level="categoria")

top_por_estado = (
  ventas.groupby(level="estado_vendedor", observed=True)
  .idxmax()
  .str[1]                      # cada valor es una tupla (estado, categoria)
)
top_por_estado.head()

estado_vendedor
BA                cool_stuff
DF    informatica_acessorios
ES        ferramentas_jardim
GO     utilidades_domesticas
MG        relogios_presentes
Name: price, dtype: object

*Notas:* la rebanada `"b":"e"` incluye la etiqueta `"e"` pero **no** `"eletronicos"`, porque en orden de texto `"eletronicos"` > `"e"`. Si quisieras todo lo que empieza con b, c, d o e, el límite sería `"b":"f"` (excluyendo luego `"f"`) o una máscara con `str[0].isin([...])`. `.str[1]` funciona sobre una Series de tuplas para extraer un elemento por posición. Una alternativa más explícita: `.map(lambda t: t[1])`.

### Solución 4

In [59]:
n_bf = len(orders_t.loc["2017-11-20":"2017-11-26"])

trim = orders_t.loc["2018"].index.to_period("Q")
por_trimestre = trim.value_counts().sort_index()
por_trimestre.index = por_trimestre.index.astype(str)

pct_fin_semana = (orders_t.index.day_of_week >= 5).mean() * 100
n_bf, por_trimestre, round(pct_fin_semana, 2)

(7,
 order_purchase_timestamp
 2018Q1    81
 2018Q2    67
 2018Q3    36
 Name: count, dtype: int64,
 np.float64(22.0))

*Nota:* con un string de sólo fecha (`"2017-11-26"`) como límite superior, pandas incluye **todo ese día** hasta las 23:59:59. Por eso no hace falta escribir `"2017-11-27"`. `day_of_week` va de lunes = 0 a domingo = 6.

### Solución 5

In [60]:
res = items_cat.groupby("estado_vendedor", observed=True)[["price", "freight_value"]].agg(["sum", "mean"])
res.columns = ["_".join(c) for c in res.columns]
res = res.assign(flete_relativo=lambda d: d["freight_value_mean"] / d["price_mean"])

estados_flete_caro = sorted(res.index[res["flete_relativo"] > 0.25].astype(str))

# Bonus: de nombres planos a MultiIndex
res_mi = res.drop(columns="flete_relativo")
res_mi.columns = res_mi.columns.str.rsplit("_", n=1, expand=True)
estados_flete_caro, res_mi.head(3)

(['ES', 'MT', 'SC'],
                   price         freight_value       
                     sum    mean           sum   mean
 estado_vendedor                                     
 BA               229.99  229.99         26.06  26.06
 DF               518.90  259.45         63.70  31.85
 ES               297.00   99.00         99.24  33.08)

*Nota del bonus:* usamos `rsplit` con `n=1` (desde la derecha, una vez) porque `freight_value` contiene un `_`. Con `split` normal obtendríamos `("freight", "value", "mean")`.

---
## Resumen del módulo

- `.loc` incluye el extremo; `.iloc` no. `.at`/`.iat` para valores sueltos.
- Indexa por la llave natural con `set_index(..., verify_integrity=True)` y **ordena** (`sort_index`).
- Las operaciones se **alinean por etiquetas**; usa `map` con Series indexadas para *lookups* seguros.
- `MultiIndex`: `.loc[(a, b)]`, selección parcial, `pd.IndexSlice`, `.xs(level=...)`, `get_level_values`.
- Agrega por nivel con `groupby(level=...)`; reorganiza con `swaplevel`, `droplevel`, `rename_axis`.
- Aplana columnas jerárquicas con `"_".join`; vuelve con `str.rsplit(..., expand=True)`.
- Un `DatetimeIndex` ordenado permite `.loc["2018-03"]` y rangos por string.

**Siguiente:** Módulo 03 · `groupby` a fondo: `agg`, `transform`, `filter`, `apply` y agregaciones con nombre.